# **Install Library**

In [36]:
# Install library

!pip install transformers
!pip install faiss-cpu
!pip install pandas
!pip install numpy
!pip install torch
!pip install streamlit
!pip install scikit-learn

# **Download Datasets**

In [37]:
#@title import kebutuhan library
import pandas as pd
import re
import torch
import faiss
import numpy as np
import os

In [38]:
if os.path.exists('data/sample_train.csv'):
    print("Menggunakan data lokal")
    df_train = pd.read_csv('data/sample_train.csv')
    df_valid = pd.read_csv('data/sample_valid.csv')
    df_test  = pd.read_csv('data/sample_test.csv')
else:
    print("Menggunakan data dari Github")
    train_url = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
    valid_url = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv"
    test_url  = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/test_preprocess.tsv"
    df_train = pd.read_csv(train_url, sep='\t', header=None, names=['text', 'label'])
    df_valid = pd.read_csv(valid_url, sep='\t', header=None, names=['text', 'label'])
    df_test  = pd.read_csv(test_url,  sep='\t', header=None, names=['text', 'label'])

Menggunakan data lokal


# **Exploratory Data Analysis (EDA)**

In [39]:
#@title show the data

def showdata(df_train, df_valid, df_test):
  for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    print(f"\n{name}")
    print(f"{display(df)}")
showdata(df_train, df_valid, df_test)


train


,text,label
0,warung ini dimiliki oleh pengusaha pabrik tahu...,positive
1,mohon ulama lurus dan k212 mmbri hujjah partai...,neutral
2,lokasi strategis di jalan sumatera bandung tem...,positive
3,betapa bahagia nya diri ini saat unboxing pake...,positive
4,duh jadi mahasiswa jangan sombong dong kasih k...,negative
...,...,...
97,biar besok jadi anggota dpr,neutral
98,tokohtokoh penggerak gerakan partai pki indone...,neutral
99,gus mus bela dan doakan menteri susi pudjiastu...,neutral
100,sudah di revisi om dengan,neutral


None

valid


,text,label
0,meski masa kampanye sudah selesai bukan berati...,neutral
1,tidak enak,negative
2,restoran ini menawarkan makanan sunda kami mem...,positive
3,lokasi di alun alun masakan padang ini cukup t...,positive
4,betapa bejad kader gerindra yang anggota dprd ...,negative
...,...,...
97,dedi mulyadi dan ridwan kamil dianggap kepala ...,neutral
98,11 kamera 3 kamera tetap 1 kamera depan dan 1 ...,neutral
99,sby penasaran hingga akhirnya mempertanyakan r...,neutral
100,arkeolog mesir temukan spinx baru di kuil kom ...,neutral


None

test


,text,label
0,kemarin gue datang ke tempat makan baru yang a...,negative
1,kayak nya sih gue tidak akan mau balik lagi ke...,negative
2,kalau dipikirpikir sebenarnya tidak ada yang b...,negative
3,ini pertama kalinya gua ke bank buat ngurusin ...,negative
4,waktu sampai dengan gue pernah disuruh ibu lat...,negative
...,...,...
97,hari ini periode submission xpresound telah di...,neutral
98,informasi tambahan bahwa untuk pembayaran rp 2...,neutral
99,sebentar lagi asian games 2018 dimulai nih cob...,neutral
100,pagi ini pemkab pemalang melalui menyelenggara...,neutral


None


In [40]:
#@title null check

def nullcheck(df_train, df_valid, df_test):
  for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    print(f"{name} = {df.isnull().sum().sum()} null values")
nullcheck(df_train, df_valid, df_test)

train = 0 null values
valid = 0 null values
test = 0 null values


In [41]:
#@title label check

def label_check(df_train, df_valid, df_test):
  for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    print(f"\n{name}")
    print(df["label"].value_counts())
label_check(df_train, df_valid, df_test)


train
label
positive    34
neutral     34
negative    34
Name: count, dtype: int64

valid
label
neutral     34
negative    34
positive    34
Name: count, dtype: int64

test
label
negative    34
positive    34
neutral     34
Name: count, dtype: int64


In [42]:
#@title avarage length of text

def avglen_check(df_train, df_valid, df_test):
    for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
        avg = df['text'].str.len().mean()
        print(f"{name} = rata-rata panjang teks: {avg:.1f} karakter")
avglen_check(df_train, df_valid, df_test)

train = rata-rata panjang teks: 146.5 karakter
valid = rata-rata panjang teks: 141.2 karakter
test = rata-rata panjang teks: 192.6 karakter


# **Preprocessing**

In [43]:
"""
Dataset SmSA dipreprocess oleh IndoNLU
namun saya akan tetap menerapkan preprocessing terlebih dahulu
untuk memastikan data konsisten
"""
def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df_train['text'] = df_train['text'].apply(preprocess)
df_valid['text'] = df_valid['text'].apply(preprocess)
df_test['text']  = df_test['text'].apply(preprocess)

samples = df_train['text'].sample(7, random_state=55).tolist()
for text in samples:
  print("\nSebelum:", text)
  print("Sesudah:", preprocess(text))


Sebelum: pdip sebut ridwan kamil menang karena berbaju merah
Sesudah: pdip sebut ridwan kamil menang karena berbaju merah

Sebelum: pengalaman yang cukup seru makan tanpa alas piring tapi hanya dengan alas daun makin nikmat makanan sunda yang sangat bervariatif dan cukup enak tips nya jangan terbawa emosi dan kalap memesan dan memilih semua makanan yang disediakan biasanya kalap kalau sudah melihat makanan macammacam begitu pilih seperlunya
Sesudah: pengalaman yang cukup seru makan tanpa alas piring tapi hanya dengan alas daun makin nikmat makanan sunda yang sangat bervariatif dan cukup enak tips nya jangan terbawa emosi dan kalap memesan dan memilih semua makanan yang disediakan biasanya kalap kalau sudah melihat makanan macammacam begitu pilih seperlunya

Sebelum: sudah di revisi om dengan
Sesudah: sudah di revisi om dengan

Sebelum: biar besok jadi anggota dpr
Sesudah: biar besok jadi anggota dpr

Sebelum: kebakaran hutan di sumbing meluas
Sesudah: kebakaran hutan di sumbing meluas

# **Tokenisasi & Embedding**

In [44]:
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p2")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p2")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def embedding(text):
  # proses tokenisasi teks
  input = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
  input = {k: v.to(device) for k, v in input.items()}

  # proses dengan model
  with torch.no_grad():
    outputs = model(**input)

  # mengambil representasi vektor (CLS token)
  embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
  return embedding

#testing
test_embedding = embedding("saya")
print("Ukuran embedding:", test_embedding.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ukuran embedding: (768,)


# **FAISS INDEX**

In [45]:
print("membuat embedding")
train_embedding = np.array([embedding(text) for text in df_train["text"]])
print("embedding berhasil dibuat!", train_embedding.shape)

membuat embedding
embedding berhasil dibuat! (102, 768)


In [46]:
#faiss index

dimension = 768
index = faiss.IndexFlatL2(dimension)

index.add(train_embedding)

print("Total vektor di FAISS:", index.ntotal)

#testing faiss

query = df_train['text'][0]
query_embedding = embedding(query).reshape(1, -1)

distances, indices = index.search(query_embedding, k=3)

print("Query:", query)
print("\nKalimat paling mirip:")
for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {df_train['text'][idx]} (jarak: {distances[0][i]:.4f})")

#simpan faiss index

faiss.write_index(index, "smsa_faiss.index")
print("\nFAISS index berhasil disimpan!")

Total vektor di FAISS: 102
Query: warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung tahu berkualitas dipadu keahlian memasak dipadu kretivitas jadilah warung yang menyajikan menu utama berbahan tahu ditambah menu umum lain seperti ayam semuanya selera indonesia harga cukup terjangkau jangan lewatkan tahu bletoka nya tidak kalah dengan yang asli dari tegal

Kalimat paling mirip:
1. warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung tahu berkualitas dipadu keahlian memasak dipadu kretivitas jadilah warung yang menyajikan menu utama berbahan tahu ditambah menu umum lain seperti ayam semuanya selera indonesia harga cukup terjangkau jangan lewatkan tahu bletoka nya tidak kalah dengan yang asli dari tegal (jarak: 0.0000)
2. rasa bakso cuanki dan batagor cukup selalu ramai pengunjung buka sekitar jam 10 pagi dari jalan riau menuju jalan a yani ada sisi kiri jalan sekitar ja

# **TRAINING**

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

#membuat embedding "valid"
valid_embeddings = np.array([embedding(text) for text in df_valid['text']])
print(valid_embeddings.shape)
#membuat embedding "test"
test_embeddings = np.array([embedding(text) for text in df_test['text']])
print(test_embeddings.shape)

#encode label
le = LabelEncoder()
train_labels = le.fit_transform(df_train['label'])
valid_labels = le.transform(df_valid['label'])
test_labels  = le.transform(df_test['label'])
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

(102, 768)
(102, 768)
Label mapping: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [48]:
#training dan evaluasi
clf = LogisticRegression(max_iter=1000)
clf.fit(train_embedding, train_labels)
print("Training selesai!")

valid_preds = clf.predict(valid_embeddings)
print("\n=== HASIL VALIDASI ===")
print(classification_report(valid_labels, valid_preds, target_names=le.classes_))

# Evaluasi Test ← hasil akhir
test_preds = clf.predict(test_embeddings)
print("\n=== HASIL TEST ===")
print(classification_report(test_labels, test_preds, target_names=le.classes_))

Training selesai!

=== HASIL VALIDASI ===
              precision    recall  f1-score   support

    negative       0.82      0.91      0.86        34
     neutral       0.91      0.85      0.88        34
    positive       0.88      0.82      0.85        34

    accuracy                           0.86       102
   macro avg       0.87      0.86      0.86       102
weighted avg       0.87      0.86      0.86       102


=== HASIL TEST ===
              precision    recall  f1-score   support

    negative       0.68      1.00      0.81        34
     neutral       1.00      0.47      0.64        34
    positive       0.69      0.74      0.71        34

    accuracy                           0.74       102
   macro avg       0.79      0.74      0.72       102
weighted avg       0.79      0.74      0.72       102



In [49]:
#sample prediksi
def predict(text):
    emb = embedding(text).reshape(1, -1)
    pred = clf.predict(emb)
    proba = clf.predict_proba(emb).max()
    label = le.inverse_transform(pred)[0]
    print(f"Teks    : {text}")
    print(f"Prediksi: {label}")
    print("\n")

# Test beberapa contoh
predict("Produk ini sangat bagus dan recommended!")
predict("Pelayanan sangat lambat dan mengecewakan")
predict("nolimit adalah sebuah perusahaan")

Teks    : Produk ini sangat bagus dan recommended!
Prediksi: positive


Teks    : Pelayanan sangat lambat dan mengecewakan
Prediksi: neutral


Teks    : nolimit adalah sebuah perusahaan
Prediksi: neutral




# **SAVE SAMPLE DATA**

In [50]:
import os
os.makedirs('data', exist_ok=True)

df_train.groupby('label').head(100).reset_index(drop=True).to_csv('data/sample_train.csv', index=False)
df_valid.groupby('label').head(100).reset_index(drop=True).to_csv('data/sample_valid.csv', index=False)
df_test.groupby('label').head(100).reset_index(drop=True).to_csv('data/sample_test.csv', index=False)

print("Semua sample berhasil disimpan!")
print(f"Train : {len(df_train.groupby('label').head(34))} baris")
print(f"Valid : {len(df_valid.groupby('label').head(34))} baris")
print(f"Test  : {len(df_test.groupby('label').head(34))} baris")

Semua sample berhasil disimpan!
Train : 102 baris
Valid : 102 baris
Test  : 102 baris
